# A5 làm lại — bước thử 200 mẫu với mã đã sửa (TASKS "A5, kết quả duyệt lần thử đầu"; HUONG_DAN_GPU_08-11). CHƯA train gì.
Lần thử đầu bị bỏ vì bộ sinh call gán id sai loại (vd. `light_set` trỏ vào điều hòa). Mã mới sửa lỗi đó. Notebook này làm đúng 5 bước của Phương: sinh 200 mẫu call (ưu tiên chuỗi dài) → **Qwen3-8B và Qwen3.5-9B viết câu** → **kiểm tra khứ hồi** bằng Qwen3-4B + adapter `q4b_lora` (kèm CONTROL trên 100 mẫu thật) → tổng hợp → 30 dòng ngẫu nhiên (cùng 30 id cho hai model). Khoảng 40 đến 60 phút GPU cho cả hai model.

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**.
**Input cần gắn (6 thứ):**
1. dataset `lifemate-code` **bản mới từ commit `962ca6c`** (có `resolve_refs`, `gen_calls --weights`, `verify_callgen --check-all`; bản cũ `404d56d` sẽ bị cell A-1 chặn)
2. dataset `train+public_test` (raw)
3. dataset `thuan-qwen3-4b` (Qwen3-4B: model của người kiểm tra khứ hồi)
4. **output của notebook `Hello_done`** (chứa `state/processed` và adapter `q4b_lora`)
5. Model **Qwen3-8B**
6. Model **Qwen3.5-9B**

**KHÔNG gắn:** output A4, A4b, A5 cũ hay `train-hist` (cùng có `state/processed`, làm bước khôi phục chọn nhầm), và không cần `phowhisper-large-thuan`. Câu nói và call là dữ liệu cuộc thi: notebook và output để **Private**, zip kết quả chỉ gửi qua Drive riêng, không đưa vào git hay chat.

**Thứ tự cell:** P-1 đến P-3, A-1 (kiểm tra mã và adapter), A-2 (tìm hai model, đối chiếu sha256), A-3 (sinh call), A-4-8b, A-4-9b (viết câu), A-5-8b, A-5-9b (khứ hồi), A-6 (tổng hợp), A-7 (đóng zip), A-8 (dọn, chạy cuối rồi Save Version).
**Gửi Phương:** `a5_retry_results.zip` (qua Drive riêng) và **ba số ở A-5 cho mỗi model** (dòng `CONTROL`, `prefilter passed`, `prefilter rejected`, đều nằm trong log và trong bản tổng hợp A-6).
Nếu một model lỗi khi nạp: báo **nguyên văn** lỗi, không tự sửa mã; A-5 và các bước sau vẫn chạy được cho model còn lại (đặt `GEN_SIZES="8b"` hoặc `"9b"` ở đầu cell A-2).

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = Qwen3-4B (hidden 2560, 36 tầng), tìm theo cấu hình vì còn có Qwen3-8B/Qwen3.5-9B trong input (không lấy "qwen3 đầu tiên")
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
QW=$(python -X utf8 - <<'EOF'
import json
from pathlib import Path
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    t = c.get("text_config", c)
    if c.get("model_type") == "qwen3" and t.get("hidden_size") == 2560 and t.get("num_hidden_layers") == 36:
        print(p.parent)
        break
EOF
)
echo "Qwen3-4B dir: ${QW:-KHÔNG THẤY}"
[ -n "$QW" ] || { echo "ERROR: không thấy Qwen3-4B (hidden 2560, 36 tầng). Gắn dataset thuan-qwen3-4b."; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ Hello_done?"; ls /kaggle/working/checkpoints /kaggle/working/checkpoints/q4b_lora | head -20

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn hai đoạn mã phụ (chỉ in số đếm tổng hợp, không in nội dung câu nói hay call)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/a5r_summary.py <<'PYEOF'
"""Tổng hợp A5 làm lại cho MỘT model viết câu: chỉ in SỐ ĐẾM (không in nội dung câu nói hay call).
Ghi thêm 30 dòng ngẫu nhiên (cùng 30 id cho mọi model) ra file để Phương duyệt.
usage: python a5r_summary.py <callgen_checked_<size>.jsonl> <size> <out_dir> [<verify_log.txt>]
Chạy từ thư mục gốc repo (cần data/raw/..., src/)."""
import json
import random
import re
import statistics
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from lifemate.callgen import app_of, text_ok  # noqa: E402
from lifemate.jsonl import read_jsonl  # noqa: E402
from lifemate.paths import split_dir  # noqa: E402
from lifemate.prompt import load_catalog  # noqa: E402

HELD = {"flights", "bike_rental", "package_tracking", "shopping_list", "podcast_radio", "loyalty"}
src, size, out_dir = Path(sys.argv[1]), sys.argv[2], Path(sys.argv[3])
log = Path(sys.argv[4]) if len(sys.argv) > 4 else None
out_dir.mkdir(parents=True, exist_ok=True)
rows = read_jsonl(src)
cat = load_catalog(split_dir("train") / "train_tools_catalog.json")
real_texts = {it["audio_text"].strip().lower() for it in read_jsonl(split_dir("train") / "train.jsonl")}
cjk = re.compile(r"[぀-ヿ㐀-鿿가-힯]")


def pct(a, b):
    return f"{a}/{b} ({100 * a / b:.1f}%)" if b else f"{a}/0"


n = len(rows)
print(f"== A5 làm lại, model viết câu {size}: {n} mẫu trong {src.name}")
for r in rows:
    r["_ok"], r["_why"] = text_ok(r["audio_text"], r, cat)
    r["_ncalls"] = len(r["label"]["tool_calls"])
ver = sum(1 for r in rows if r.get("verified"))
tok = sum(r["_ok"] for r in rows)
pre = sum(1 for r in rows if r.get("prefilter_ok"))
ag = sum(1 for r in rows if r.get("agent_ok"))
rt = sum(1 for r in rows if r.get("roundtrip"))
print(f"model tự chấm CÓ (verified)           : {pct(ver, n)}")
print(f"qua luật text_ok (bản mới)            : {pct(tok, n)}")
print(f"qua bộ lọc rẻ (verified và text_ok)   : {pct(pre, n)}")
print(f"agent q4b_lora đạt khứ hồi (agent_ok) : {pct(ag, n)}   <- trên TẤT CẢ mẫu (--check-all)")
print(f"được giữ (lọc rẻ VÀ khứ hồi)         : {pct(rt, n)}")
pre_rows = [r for r in rows if r.get("prefilter_ok")]
rej_rows = [r for r in rows if not r.get("prefilter_ok")]
print(f"agent_ok trong mẫu QUA lọc rẻ         : {pct(sum(bool(r.get('agent_ok')) for r in pre_rows), len(pre_rows))}")
print(f"agent_ok trong mẫu BỊ LOẠI bởi lọc rẻ : {pct(sum(bool(r.get('agent_ok')) for r in rej_rows), len(rej_rows))}")
print("lý do text_ok loại:", dict(Counter(r["_why"] for r in rows).most_common()))
if log and log.is_file():
    print("-- trích từ log verify_callgen (CONTROL và hai tỉ lệ cuối):")
    for line in log.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith(("CONTROL", "prefilter")):
            print("  " + line)

print("\n-- theo số call (tổng | qua lọc rẻ | agent_ok | được giữ):")
for k in sorted({r["_ncalls"] for r in rows}):
    g = [r for r in rows if r["_ncalls"] == k]
    print(f"  {k} call: {len(g):3} | {sum(bool(r.get('prefilter_ok')) for r in g):3} | {sum(bool(r.get('agent_ok')) for r in g):3} | {sum(bool(r.get('roundtrip')) for r in g):3}")

words = sorted(len(r["audio_text"].split()) for r in rows)
print(f"\nsố từ mỗi câu: min {words[0]}, trung vị {statistics.median(words)}, p90 {words[int(0.9 * (len(words) - 1))]}, max {words[-1]}")
print(f"câu rỗng: {sum(1 for r in rows if not r['audio_text'].strip())} | có ký tự Hán/Nhật/Hàn: {sum(bool(cjk.search(r['audio_text'])) for r in rows)} "
      f"(trong số được giữ: {sum(bool(cjk.search(r['audio_text'])) for r in rows if r.get('roundtrip'))})")
print(f"câu trùng y hệt một câu THẬT trong train: {pct(sum(1 for r in rows if r['audio_text'].strip().lower() in real_texts), n)}")
print(f"câu trùng nhau trong {n} mẫu: {n - len({r['audio_text'].strip().lower() for r in rows})}")
print("số app mỗi mẫu:", dict(sorted(Counter(len({app_of(c['name']) for c in r['label']['tool_calls']}) for r in rows).items())))
print("app nhiều nhất:", dict(Counter(a for r in rows for a in {app_of(c['name']) for c in r['label']['tool_calls']}).most_common(8)))
held = [r for r in rows if {app_of(c["name"]) for c in r["label"]["tool_calls"]} & HELD]
print(f"mẫu dùng 1 trong 6 app A4 giấu: {pct(len(held), n)} (được giữ: {sum(bool(r.get('roundtrip')) for r in held)})")
print(f"mẫu có refs (mục id trỏ tới): {pct(sum(1 for r in rows if r.get('refs')), n)}")

# 30 dòng: cùng 30 id cho mọi model (chọn từ id đã sắp xếp, seed 0), kèm cờ để Phương thấy bộ lọc có bắt được câu xấu không
ids = sorted(r["id"] for r in rows)
pick_ids = set(random.Random(0).sample(ids, min(30, n)))
pick = sorted((r for r in rows if r["id"] in pick_ids), key=lambda r: r["id"])
slim = [{"id": r["id"], "audio_text": r["audio_text"], "tool_calls": r["label"]["tool_calls"], "refs": r.get("refs", {}),
         "verified": r.get("verified"), "text_ok": r["_ok"], "text_ok_reason": r["_why"], "agent_ok": r.get("agent_ok")} for r in pick]
with open(out_dir / f"a5r_sample30_{size}.jsonl", "w", encoding="utf-8", newline="\n") as f:
    for s in slim:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
with open(out_dir / f"a5r_sample30_{size}.txt", "w", encoding="utf-8", newline="\n") as f:
    for i, s in enumerate(slim, 1):
        f.write(f"#{i} id={s['id']}  verified={s['verified']}  text_ok={s['text_ok']} ({s['text_ok_reason']})  agent_ok={s['agent_ok']}\n")
        f.write(f"  CÂU : {s['audio_text']}\n")
        for c in s["tool_calls"]:
            f.write(f"  CALL: {json.dumps(c, ensure_ascii=False)}\n")
        for k, v in s["refs"].items():
            f.write(f"  REF : {k} -> {json.dumps(v, ensure_ascii=False)[:240]}\n")
        f.write("\n")
print(f"đã ghi 30 dòng ngẫu nhiên (seed 0, cùng id cho mọi model) -> {out_dir}/a5r_sample30_{size}.txt và .jsonl (KHÔNG in nội dung ở đây)")
PYEOF
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# A-1: kiểm tra mã mới, adapter q4b_lora (người kiểm tra khứ hồi) và dev_ids. Thiếu hoặc sai thì DỪNG.
# Rồi dọn BẢN SAO các adapter không dùng mà setup đã chép từ Hello_done (bản gốc vẫn nằm trong input).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/gen_calls.py
need scripts/gen_text.py
need scripts/verify_callgen.py
need src/lifemate/callgen.py
need data/processed/dev_ids.txt
need checkpoints/q4b_lora/adapter_model.safetensors
need checkpoints/q4b_lora/adapter_config.json
need models.lock.json
need "data/raw/train+public_test/train/train.jsonl"
grep -q "def resolve_refs" src/lifemate/callgen.py 2>/dev/null && echo "ok     code là bản MỚI (có resolve_refs)" || { echo "THIẾU  code còn là bản CŨ: upload lifemate-code mới (từ commit 962ca6c)"; MISSING=1; }
grep -q -- "--weights" scripts/gen_calls.py 2>/dev/null && echo "ok     gen_calls có --weights" || { echo "THIẾU  gen_calls chưa có --weights (code cũ)"; MISSING=1; }
grep -q -- "--check-all" scripts/verify_callgen.py 2>/dev/null && echo "ok     verify_callgen có --check-all" || { echo "THIẾU  verify_callgen chưa có --check-all (code cũ)"; MISSING=1; }
if [ "$MISSING" = 1 ]; then echo "-> Kiểm tra input: lifemate-code (bản từ 962ca6c), train+public_test, thuan-qwen3-4b, output notebook Hello_done."; exit 1; fi
n=$(wc -l < data/processed/dev_ids.txt)
[ "$n" = 155 ] || { echo "dev_ids.txt có $n dòng, phải là 155"; exit 1; }
echo "dev_ids.txt: 155 id (đúng)"
h=$(sha256sum checkpoints/q4b_lora/adapter_model.safetensors | cut -d' ' -f1)
want=$(grep -o '"adapter_model.safetensors": *"[0-9a-f]*"' models.lock.json | grep -o '[0-9a-f]\{64\}' | head -1)
echo "q4b_lora sha256 : $h"
echo "models.lock.json: $want"
if [ "$h" = "$want" ]; then echo "q4b_lora: TRÙNG adapter của bài nộp"; else echo "CẢNH BÁO: q4b_lora KHÁC adapter ghi trong models.lock.json. Báo lại Phương, vẫn chạy tiếp được nhưng CONTROL có thể khác."; fi
echo "-- dọn bản sao không dùng"
du -sh checkpoints | cut -f1 | xargs echo "trước khi dọn:"
for d in checkpoints/*; do [ "$(basename "$d")" = "q4b_lora" ] || rm -rf "$d"; done
rm -rf checkpoints/q4b_lora/checkpoint-*
rm -rf /kaggle/working/outputs/*
mkdir -p outputs/logs
rm -f data/processed/callgen_*.jsonl
du -sh checkpoints | cut -f1 | xargs echo "sau khi dọn:"
ls checkpoints; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A-2: tìm cả hai model viết câu trong /kaggle/input theo cấu hình (không đoán tên thư mục) và đối chiếu sha256 với bản gốc Hugging Face.
# Mất khoảng 3 đến 6 phút (đọc 36 GB để tính sha256). Chỉ thử một model: đặt GEN_SIZES="8b" (hoặc "9b") ở dòng dưới.
export GEN_SIZES=${GEN_SIZES:-"8b 9b"}
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
python -X utf8 - <<'EOF'
import hashlib
import json
import os
import sys
from pathlib import Path

# (model_type, hidden_size, số tầng văn bản). Qwen3.5 là model đa phương thức: cấu hình văn bản nằm trong "text_config".
WANT = {"8b": ("qwen3", 4096, 36), "9b": ("qwen3_5", 4096, 32), "14b": ("qwen3", 5120, 40), "32b": ("qwen3", 5120, 64)}
# sha256 các shard của bản gốc trên Hugging Face (Qwen/Qwen3-8B và Qwen/Qwen3.5-9B), tra theo kích thước file vì Kaggle có thể đổi tên shard.
OFFICIAL = {
    "8b": {3996250744: "31d6a825ae35f11fb85b195b4c42c146c051e446433125a215336abdf95cbf5f",
           3993160032: "5991236cea6fe21f3d43cab0f0e84448734fbbe0789816202989f2ddc9d18282",
           3959604768: "c5185c4794be2d8a9784d5753c9922db38df478ce11f9ed0b415b7304d896836",
           3187841392: "b5ee7de71fbf17db3d5704e0c8f2bc7d005ca9e1d7ca2aeb19827b0cfcaa917a",
           1244659840: "20c2d6366ab85c90786ccdd829cd2b9e7d30ef3b2ebbb998280e7e4014b542ff"},
    "9b": {5276436216: "db6f444b43d318c92f360a13a25561a6a65b10c0631b8ed305a426dbaa6c380e",
           5335161512: "31c7d7e2dd5d207840b31cc59083c8f4c4718959149e0358c0364052bb9a0330",
           5368717440: "7ec36ba3a4176a44c3c0876ad80c56a2f70c84bf008d82e9501df642f17dadec",
           3325995712: "b62b0c4cd7e44edee103ee8f4fe225f246d5e768e07bfd5f25b63a8aa1fdd0c6"},
}
sizes = os.environ["GEN_SIZES"].split()
found = []
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    if not str(c.get("model_type", "")).startswith("qwen3"):
        continue
    t = c.get("text_config", c)
    weights = sorted(p.parent.glob("*.safetensors"))
    found.append((p.parent, c.get("model_type"), t.get("hidden_size"), t.get("num_hidden_layers"), len(weights),
                  sum(w.stat().st_size for w in weights) / 1e9))
print("Các model qwen3* đang gắn vào notebook:")
for d, mt, h, L, nw, gb in found:
    print(f"  {d}  type={mt} hidden={h} layers={L}  {nw} file safetensors, {gb:.1f} GB")

env_lines, info = [], []
problems = []
for size in sizes:
    want = WANT[size]
    pick = [f for f in found if (f[1], f[2], f[3]) == want and f[4] > 0]
    if not pick:
        problems.append(f"KHÔNG thấy model {size} (type={want[0]}, hidden={want[1]}, layers={want[2]}) có safetensors. "
                        "Add Input > Models > tìm đúng model rồi chạy lại cell này.")
        continue
    d = pick[0][0]
    env_lines.append(f'export GEN_{size.upper()}="{d}"')
    info.append(f"== GEN_SIZE={size}\nthư mục: {d}\nconfig.json sha256: {hashlib.sha256((d / 'config.json').read_bytes()).hexdigest()}\n"
                f"model_type={pick[0][1]} hidden_size={pick[0][2]} num_hidden_layers={pick[0][3]} tổng safetensors={pick[0][5]:.2f} GB")
    ref = OFFICIAL.get(size)
    if ref:
        verdict, matched = [], set()
        shards = sorted(d.glob("*.safetensors"))
        for w in shards:
            sz = w.stat().st_size
            if sz not in ref:
                verdict.append(f"{w.name}: kích thước KHÁC bản gốc (không đối chiếu được)")
                continue
            h = hashlib.sha256()
            with open(w, "rb") as fh:
                while block := fh.read(1 << 24):
                    h.update(block)
            ok = h.hexdigest() == ref[sz]
            if ok:
                matched.add(sz)
            verdict.append(f"{w.name}: " + ("TRÙNG bản gốc" if ok else "KHÁC bản gốc (sha256 không trùng)"))
        full = matched == set(ref) and len(shards) == len(ref)
        head = (f"ĐỐI CHIẾU với bản gốc Hugging Face: {len(matched)}/{len(ref)} shard trùng sha256 (đủ, là bản gốc)" if full else
                f"ĐỐI CHIẾU với bản gốc Hugging Face: CHƯA XÁC MINH ({len(matched)}/{len(ref)} shard trùng)")
        info.append(head + "\n" + "\n".join("  " + v for v in verdict))
        print(f"{size}: {head}")
    print(f"{size}: DÙNG {d}")
if problems:
    sys.exit("\n".join(problems))
Path("/kaggle/working/a5_env.sh").write_text("\n".join(env_lines) + f'\nexport GEN_SIZES="{" ".join(sizes)}"\n', encoding="utf-8")
Path("/kaggle/working/gen_model_info.txt").write_text("\n\n".join(info) + "\n", encoding="utf-8")
EOF
cat /kaggle/working/a5_env.sh

In [ ]:
%%bash
# A-3 (CPU, vài giây): sinh 200 mẫu call bằng MÃ MỚI (id cùng loại, nhất quán trong mẫu), ưu tiên chuỗi dài theo hướng dẫn của Phương.
# Seed 42 cố định nên cả hai model nhận đúng 200 mẫu giống nhau.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 scripts/gen_calls.py --n 200 --seed 42 --weights 1:0.3,2:0.4,3:0.3 2>&1 | tee outputs/logs/a5r_gen_calls.txt
wc -l data/processed/callgen_pending.jsonl

In [ ]:
%%bash
# A-4-8b (GPU): model 8b viết câu cho 200 mẫu rồi tự chấm CÓ/KHÔNG (bước 3 của Phương). Chạy trên máy Kaggle, không gọi API ngoài.
# Nếu lỗi nạp model (nhất là Qwen3.5-9B): báo NGUYÊN VĂN dòng "AutoModelForCausalLM failed ..." hoặc traceback, không tự sửa mã.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
SIZE=8b
VAR=GEN_${SIZE^^}
GEN=${!VAR}
echo "model: $GEN ($SIZE)"
OUT=data/processed/callgen_text_${SIZE}.jsonl
rm -f "$OUT"
START=$(date +%s)
python -X utf8 scripts/gen_text.py --model "$GEN" --inp data/processed/callgen_pending.jsonl --out "$OUT" --batch 32 2>&1 | tee outputs/logs/a5r_gen_text_${SIZE}.txt
echo "thời gian gen_text: $(( $(date +%s) - START )) s (gồm nạp model)" | tee -a outputs/logs/a5r_gen_text_${SIZE}.txt
wc -l "$OUT"

In [ ]:
%%bash
# A-4-9b (GPU): model 9b viết câu cho 200 mẫu rồi tự chấm CÓ/KHÔNG (bước 3 của Phương). Chạy trên máy Kaggle, không gọi API ngoài.
# Nếu lỗi nạp model (nhất là Qwen3.5-9B): báo NGUYÊN VĂN dòng "AutoModelForCausalLM failed ..." hoặc traceback, không tự sửa mã.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
SIZE=9b
VAR=GEN_${SIZE^^}
GEN=${!VAR}
echo "model: $GEN ($SIZE)"
OUT=data/processed/callgen_text_${SIZE}.jsonl
rm -f "$OUT"
START=$(date +%s)
python -X utf8 scripts/gen_text.py --model "$GEN" --inp data/processed/callgen_pending.jsonl --out "$OUT" --batch 32 2>&1 | tee outputs/logs/a5r_gen_text_${SIZE}.txt
echo "thời gian gen_text: $(( $(date +%s) - START )) s (gồm nạp model)" | tee -a outputs/logs/a5r_gen_text_${SIZE}.txt
wc -l "$OUT"

In [ ]:
%%bash
# A-5-8b (GPU): kiểm tra khứ hồi tệp câu của 8b bằng Qwen3-4B + adapter q4b_lora làm người kiểm tra (bước 4 của Phương).
# In 3 số cần gửi: CONTROL (100 mẫu thật, mức trần), tỉ lệ đạt của mẫu QUA lọc rẻ, và của mẫu BỊ LOẠI bởi lọc rẻ.
# Khoảng 8 đến 20 phút. Log chỉ hiện sau khi cell xong; chạy tương tác thì dòng tiến độ hiện mỗi 50 mẫu.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
SIZE=8b
IN=data/processed/callgen_text_${SIZE}.jsonl
OUT=data/processed/callgen_checked_${SIZE}.jsonl
[ -s "$IN" ] || { echo "THIẾU $IN: chạy cell A-4-8b trước"; exit 1; }
rm -f "$OUT"
START=$(date +%s)
python -X utf8 scripts/verify_callgen.py --model "$QWEN" --adapter checkpoints/q4b_lora --inp "$IN" --out "$OUT" \
  --check-all --control data/processed/dev_ids.txt --control-n 100 2>&1 | tee outputs/logs/a5r_verify_${SIZE}.txt
echo "thời gian verify_callgen: $(( $(date +%s) - START )) s" | tee -a outputs/logs/a5r_verify_${SIZE}.txt
wc -l "$OUT"

In [ ]:
%%bash
# A-5-9b (GPU): kiểm tra khứ hồi tệp câu của 9b bằng Qwen3-4B + adapter q4b_lora làm người kiểm tra (bước 4 của Phương).
# In 3 số cần gửi: CONTROL (100 mẫu thật, mức trần), tỉ lệ đạt của mẫu QUA lọc rẻ, và của mẫu BỊ LOẠI bởi lọc rẻ.
# Khoảng 8 đến 20 phút. Log chỉ hiện sau khi cell xong; chạy tương tác thì dòng tiến độ hiện mỗi 50 mẫu.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
SIZE=9b
IN=data/processed/callgen_text_${SIZE}.jsonl
OUT=data/processed/callgen_checked_${SIZE}.jsonl
[ -s "$IN" ] || { echo "THIẾU $IN: chạy cell A-4-9b trước"; exit 1; }
rm -f "$OUT"
START=$(date +%s)
python -X utf8 scripts/verify_callgen.py --model "$QWEN" --adapter checkpoints/q4b_lora --inp "$IN" --out "$OUT" \
  --check-all --control data/processed/dev_ids.txt --control-n 100 2>&1 | tee outputs/logs/a5r_verify_${SIZE}.txt
echo "thời gian verify_callgen: $(( $(date +%s) - START )) s" | tee -a outputs/logs/a5r_verify_${SIZE}.txt
wc -l "$OUT"

In [ ]:
%%bash
# A-6: tổng hợp bằng số đếm cho từng model và ghi 30 dòng ngẫu nhiên (CÙNG 30 id cho hai model) ra file để gửi Phương. KHÔNG in nội dung câu.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
for SIZE in $GEN_SIZES; do
  if [ -s data/processed/callgen_checked_${SIZE}.jsonl ]; then
    python -X utf8 /kaggle/working/helpers/a5r_summary.py data/processed/callgen_checked_${SIZE}.jsonl $SIZE /kaggle/working/a5r_out outputs/logs/a5r_verify_${SIZE}.txt 2>&1 | tee outputs/logs/a5r_summary_${SIZE}.txt
    echo
  else
    echo "THIẾU callgen_checked_${SIZE}.jsonl: chạy A-5 cho $SIZE trước"
  fi
done
ls -la /kaggle/working/a5r_out

In [ ]:
%%bash
# A-7: gói kết quả nhỏ để gửi Phương: a5_retry_results.zip (câu đã sinh và cờ kiểm tra của cả hai model, 30 dòng để duyệt, log, môi trường, thông tin model)
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a5_retry_results.zip \
  data/processed/callgen_checked_*.jsonl \
  /kaggle/working/a5r_out/a5r_sample30_*.txt /kaggle/working/a5r_out/a5r_sample30_*.jsonl \
  outputs/logs/a5r_*.txt /kaggle/working/torch_cuda.txt /kaggle/working/gen_model_info.txt
ls -la /kaggle/working/*.zip

In [ ]:
%%bash
# A-8 (CUỐI CÙNG): dọn thư mục làm việc (xóa lifemate/ có symlink tới dữ liệu đọc-chỉ và bản sao adapter), giữ zip, câu đã sinh và log. Rồi Save Version.
cd /kaggle/working
mkdir -p a5r_keep
cp lifemate/data/processed/callgen_*.jsonl a5r_keep/ 2>/dev/null || true
rm -rf lifemate helpers setup_patched.sh config_paths.txt checkpoints
du -sh /kaggle/working/* 2>/dev/null
echo "xong. Bây giờ Save Version (private). Lấy a5_retry_results.zip ở tab Output."